# Process

Process scans (HTR and OCR)

## 1. HTR with Claude

In [ ]:
import anthropic
from datetime import datetime
from dotenv import load_dotenv
import json
import os
from pathlib import Path
import regex

In [ ]:
load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

In [ ]:
def clear_claude_storage():
    files = client.beta.files.list()
    for file in files:
        client.beta.files.delete(file.id)

In [ ]:
def save_json(results_json, name="output", suffix="_" + datetime.strftime(datetime.now(), "%Y%m%d")):
    with open(f"{name}{suffix}.json", "w", encoding='utf-8') as f:
        json.dump(results_json, f, ensure_ascii=False, indent=2)

In [ ]:
def read_json(file_name):
    with open(file_name, "r", encoding='utf-8') as infile:
        return json.load(infile)

In [ ]:
def send_claude_prompt(prompt, upload_file_name):
    try:
        upload_response = client.beta.files.upload(file=Path(upload_file_name))
        message = client.beta.messages.create(
            model="claude-sonnet-5",
            max_tokens=1024,
            messages=[
                {"role": "user", 
                 "content": [
                    {"type": "image",
                     "source": {"type": "file",
                                "file_id": upload_response.id
                               }
                    },
                    {"type": "text", "text": prompt}
                  ]}],
            betas=["files-api-2025-04-14"]
        )
    finally:
        clear_claude_storage()
    text_blocks = [block.text for block in message.content if block.type == "text"]
    return "\n".join(text_blocks)

In [ ]:
def add_comment(person_dict, comment_prefix, comment_suffix):
    if comment_prefix:
        if comment_suffix:
            person_dict["comment"] = " ".join([comment_prefix, comment_suffix])
        else:
            person_dict["comment"] = comment_prefix
    elif comment_suffix:
        person_dict["comment"] = comment_suffix

In [ ]:
def add_page_number(person_dict, page_number, sample_file_name):
    sample_file_name_parts = regex.split(r"[_.]", sample_file_name)
    sample_file_name_parts[-2] = str(page_number).zfill(len(sample_file_name_parts[-2]))
    sample_file_name_parts[-2] += "." + sample_file_name_parts[-1]
    sample_file_name_parts.pop()
    person_dict["scan_file"] = "_".join(sample_file_name_parts)
    person_dict["page_number"] = page_number

In [ ]:
def str2dict(string, page_number, sample_file_name):
    groups = regex.search(r"^(.*)```json(.*)```(.*)$", string.strip(), flags=regex.DOTALL)
    person_dict = json.loads(groups.group(2))
    add_comment(person_dict, groups.group(1).strip(), groups.group(1).strip())
    add_page_number(person_dict, page_number, sample_file_name)
    return person_dict

In [ ]:
def claude2json(results):
    results_json = []
    for page_number, result in results.items():
        results_json.append(str2dict(result, page_number, "sample_file_name"))
    return results_json

In [ ]:
int(file_name[-14:-9])

In [ ]:
source_dir = "../memories_crawl/scans/bhic/Werkendam/deel_1924-1927/out_L/selected"
last_processed = "MFF-Werkendam-1924-1927-03-00351_L_45.jpg"
results = {}
for file_name in sorted(os.listdir(source_dir)):
    if file_name <= last_processed:
        continue
    file_name_with_dir = os.path.join(source_dir, file_name)
    page_nbr = int(file_name[-14:-9])

    prompt = f"""Dear Claude, attached you will find a scan which contains the phrase 
'11. Opgave van den staat des boedels' in the top left. To the right of the phrase,
you will find some numbers denoting money values, one above each other. Usually there
are three numbers but sometimes also four, two or zero. The numbers can be preceded 
by an `f` for Dutch guilders or a double quote indicating a copy of the element above 
it (usually the `f`). We do not need these characters. Most numbers contain a decimal 
marker which could be a comma or a period. Some numbers have a hyphen behind the 
decimal marker, which stands for zero cents. A few numbers finish with a superscript 
`5` indicating half a cent. Note that there is a relation between the numbers:
number 1 equals number 2 plus number 3. Please return a line with a name of the file, 
which is: {file_name}, the model name which is 'claude-sonnet-5' and the numbers, 
separated by single spaces, and nothing else. If there are no numbers on the page, 
just return the file name and the model name."""

    results[page_nbr] = send_claude_prompt(prompt, file_name_with_dir)
    print(results[page_nbr])
    
results_json = claude2json(results)
save_json(results_json)